# 00 — Divisão oficial do corpus

Este notebook gera **uma única vez** o arquivo `data/split.csv`, com a divisão oficial usada por **todos** os modelos do projeto (tradicionais e de Deep Learning).

- Corpus: Fake.br, versão **normalizada por tamanho** (ver `docs/adr/0001-texto-normalizado-por-tamanho.md`)
- Divisão **por par alinhado**, 70% treino / 15% validação / 15% teste, `random_state=42` (ver `docs/adr/0002-divisao-por-par-alinhado.md`)

> ⚠️ O `split.csv` já está versionado no repositório. **Não rode a célula de geração de novo** a não ser que o grupo decida refazer a divisão. Os outros notebooks só precisam de `carregar_corpus()`.

## Setup (Colab ou local)

In [ ]:
import os, sys

NO_COLAB = "google.colab" in sys.modules
REPO = "LeanderHeitor/fakebr-classificacao"

if NO_COLAB and not os.path.exists("/content/fakebr-classificacao"):
    # Repositório privado: crie um token em GitHub > Settings > Developer settings
    # e salve nos Secrets do Colab (ícone de chave) com o nome GITHUB_TOKEN.
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q https://{token}@github.com/{REPO}.git /content/fakebr-classificacao

if NO_COLAB:
    %cd /content/fakebr-classificacao

## Gerar a divisão (só se `data/split.csv` não existir)

In [ ]:
from src.dados import gerar_divisao, carregar_corpus, ARQ_SPLIT

if ARQ_SPLIT.exists():
    print(f"{ARQ_SPLIT.name} já existe — mantendo a divisão oficial.")
else:
    gerar_divisao().to_csv(ARQ_SPLIT, index=False)
    print(f"{ARQ_SPLIT.name} gerado.")

## Verificações

In [ ]:
import pandas as pd

df = carregar_corpus()
print(len(df), "notícias,", df.par.nunique(), "pares")
df.groupby(["split", "rotulo"]).size().unstack().rename(columns={0: "verdadeira", 1: "falsa"})

In [ ]:
assert (df.groupby("par").split.nunique() == 1).all(), "Há par dividido entre conjuntos!"
assert (df.groupby("split").rotulo.mean() == 0.5).all(), "Conjunto desbalanceado!"
print("OK: nenhum par dividido e todos os conjuntos 50/50.")

Tamanho médio (palavras) por classe — deve ser praticamente igual, já que o texto é normalizado por tamanho:

In [ ]:
df.assign(palavras=df.texto.str.split().str.len()).groupby("rotulo").palavras.mean().rename({0: "verdadeira", 1: "falsa"})

Distribuição das categorias por conjunto. Os pares 697 e 1468 não têm metadados no corpus original e aparecem como `desconhecida`.

In [ ]:
pd.crosstab(df.categoria, df.split, margins=True)